# Урок 15 · Медленная версия: как данные попадают в нейросеть

**Сегодня НЕ строим модель.** Сегодня разбираемся с тем, что происходит *до* модели —
откуда берутся картинки, почему их делят на 255 и что значит форма `(32, 32, 3)`.

Каждую клетку кода **читай сверху вниз** и запускай по одной (Shift+Enter).
После каждого запуска — смотри на результат и отвечай на вопрос в тексте.

> 💡 Правило урока: не понял, что вывела клетка — не иди дальше. Спроси или перечитай комментарий.

## Шаг 1. Загружаем картинки

CIFAR-10 — это 60 000 маленьких цветных картинок 32×32 пикселя: самолёты, кошки, машины и ещё 7 категорий.
Он уже встроен в Keras, качать вручную ничего не надо.

In [ ]:
# импортируем инструмент для загрузки готовых датасетов
from tensorflow.keras.datasets import cifar10

# загружаем данные. Они УЖЕ разделены на обучающие (train) и тестовые (test)
# X — сами картинки, y — правильные ответы (метки: 0=самолёт, 3=кошка, ...)
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# смотрим, СКОЛЬКО у нас картинок и какой они формы
print("Картинок для обучения:", X_train.shape)
print("Картинок для теста:   ", X_test.shape)

**❓ Вопрос 1.** Клетка вывела `(50000, 32, 32, 3)`.
Что означает каждое из четырёх чисел? Запиши своими словами:

- `50000` → ...
- `32` → ...
- `32` → ...
- `3` → ...

<details><summary>Подсказка (открой, только если застрял)</summary>

50000 картинок · высота 32 пикселя · ширина 32 пикселя · 3 цвета (R, G, B).
</details>

## Шаг 2. Смотрим на ОДНУ картинку

Компьютер не видит котика. Он видит таблицу чисел. Убедимся в этом.

In [ ]:
import matplotlib.pyplot as plt

# берём самую первую картинку из обучающего набора
first_image = X_train[0]

# показываем её глазами
plt.imshow(first_image)
plt.title("Так картинку видим МЫ")
plt.axis("off")
plt.show()

# а теперь — так, как её видит компьютер: числа
print("А так её видит КОМПЬЮТЕР (кусочек 5x5 из красного канала):")
print(first_image[:5, :5, 0])

**❓ Вопрос 2.** Числа в таблице — в каком диапазоне? Найди самое маленькое и самое большое.

Впиши код сам (замени `...`):

In [ ]:
# .min() находит минимум, .max() — максимум
print("Минимальное число в картинке:", first_image....)   # ← допиши: min()
print("Максимальное число в картинке:", first_image....)   # ← допиши: max()

Должно получиться примерно от `0` до `255`.
Это яркость: `0` — цвета совсем нет, `255` — максимум.

## Шаг 3. Нормализация — самый важный шаг

Нейросети тяжело работать с числами 0–255. Ей удобнее, когда числа маленькие: от 0 до 1.
Чтобы превратить 0–255 → 0–1, надо просто **разделить на 255**.

Это и есть та самая строчка `X_train / 255.0`, которую вы видели на прошлом уроке.

In [ ]:
# делим КАЖДОЕ число во всех картинках на 255
# было: 0...255   стало: 0.0...1.0
X_train_norm = X_train / 255.0
X_test_norm = X_test / 255.0

# проверяем, что диапазон изменился
print("До нормализации:  от", X_train.min(), "до", X_train.max())
print("После нормализации: от", X_train_norm.min(), "до", X_train_norm.max())

**❓ Вопрос 3.** Картинка на экране изменится, если её нормализовать?
Сначала подумай, потом запусти клетку ниже и проверь догадку.

In [ ]:
# показываем нормализованную картинку рядом с обычной
fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.imshow(X_train[0]);      ax1.set_title("0-255");   ax1.axis("off")
ax2.imshow(X_train_norm[0]); ax2.set_title("0.0-1.0"); ax2.axis("off")
plt.show()
# Глазами картинки одинаковые! Изменились только числа, а не смысл.

## Шаг 4. Что за метки (ответы)?

`y_train` — это правильные ответы. Каждое число — это категория.

In [ ]:
# названия категорий по порядку (0-9)
classes = ["самолёт","машина","птица","кошка","олень",
           "собака","лягушка","лошадь","корабль","грузовик"]

# смотрим ответ для первой картинки
label = y_train[0][0]          # y_train[0] это [6], берём число из списка
print("Метка первой картинки:", label)
print("Это значит:", classes[label])

# покажем картинку с её подписью
plt.imshow(X_train_norm[0])
plt.title("Это: " + classes[label])
plt.axis("off")
plt.show()

---
## 🎯 Задания (выбери по силам)

### 🟢 Базовый уровень
Покажи **первые 9 картинок** обучающего набора с их подписями.
Заготовка ниже — допиши, что помечено `...`

In [ ]:
plt.figure(figsize=(6,6))
for i in range(9):
    plt.subplot(3, 3, i+1)              # сетка 3x3
    plt.imshow(X_train_norm[...])       # ← допиши: i
    plt.title(classes[y_train[...][0]]) # ← допиши: i
    plt.axis("off")
plt.tight_layout()
plt.show()

### 🟡 Продвинутый уровень
Посчитай, **сколько картинок каждой категории** в обучающем наборе.
Подсказка: пригодится `numpy` и `unique`.

In [ ]:
import numpy as np
values, counts = np.unique(y_train, return_counts=True)
for v, c in zip(values, counts):
    print(classes[v], "—", c, "картинок")

### ⭐ Со звёздочкой
Найди в наборе **любую картинку кошки** (метка 3) и покажи её.
Тебе нужно: пройти по меткам, найти первую, где метка == 3, и показать эту картинку.

In [ ]:
# твой код здесь
# подсказка: for i in range(len(y_train)): if y_train[i][0] == 3: ...


---
## Мини-итог урока

Заполни своими словами:

- Картинка для компьютера — это ...
- Форма `(32, 32, 3)` означает ...
- Нормализация — это ..., и нужна она потому что ...

> После этого урока ты **сам** можешь подготовить любой набор картинок к обучению.
> На следующем уроке возьмём эти готовые данные и построим на них CNN.